# Almond Tree Segmentation Pipeline (Improved)

In [ ]:
import os
import cv2
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from pycocotools.coco import COCO
import albumentations as A
from albumentations.pytorch import ToTensorV2
import segmentation_models_pytorch as smp
from tqdm import tqdm
import glob


# Device setup
device = torch.device("mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Custom COCO Dataset
class COCOSegmentationDataset(Dataset):
    def __init__(self, img_dir, ann_path, transform=None):
        self.img_dir = img_dir
        self.coco = COCO(ann_path)
        self.image_ids = list(self.coco.imgs.keys())
        self.transform = transform

    def __len__(self):
        return len(self.image_ids)

    def __getitem__(self, idx):
        img_id = self.image_ids[idx]
        img_info = self.coco.loadImgs(img_id)[0]
        path = os.path.join(self.img_dir, img_info['file_name'])
        image = cv2.imread(path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        ann_ids = self.coco.getAnnIds(imgIds=img_id)
        anns = self.coco.loadAnns(ann_ids)

        mask = np.zeros((img_info['height'], img_info['width']), dtype=np.uint8)
        for ann in anns:
            mask = np.maximum(mask, self.coco.annToMask(ann))

        if self.transform:
            augmented = self.transform(image=image, mask=mask)
            image = augmented['image']
            mask = augmented['mask'].unsqueeze(0).float()

        return image, mask

### Transformations

In [ ]:
# Updated train transform
train_transform = A.Compose([
    A.Resize(1024, 1024),
    A.HorizontalFlip(p=0.2),
    A.ShiftScaleRotate(shift_limit=0.01, 
                       scale_limit=0.05, 
                       rotate_limit=5, p=0.2),
    A.Normalize(),
    ToTensorV2(),
])

# Validation transform remains minimal
val_transform = A.Compose([
    A.Resize(1024, 1024),
    A.Normalize(),
    ToTensorV2(),
])


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import cv2
from torchvision.transforms import ToTensor
from torch.utils.data import Dataset

# Load a few raw images from your training set folder (update the path)
image_paths = sorted(glob.glob("/Users/snirtahasa/Desktop/Almonds - Project/Almons-Trees-3/train/*.jpg"))[:5]  # Limit to first 5

# Define Albumentations transforms
original_transform = A.Compose([
    A.Resize(1024, 1024),
])

augmented_transform = A.Compose([
    A.Resize(1024, 1024),                      # Uniform size
    A.HorizontalFlip(p=0.2),                   # Minimal flipping
    A.ShiftScaleRotate(shift_limit=0.01, 
                       scale_limit=0.05, 
                       rotate_limit=5, p=0.2), # Gentle augmentations
    A.Normalize(),                             # Essential for model input
    ToTensorV2(),
])

def show_augmented_images(image_paths):
    for img_path in image_paths:
        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        # Apply transforms
        orig_img = original_transform(image=image)["image"]
        aug_img = augmented_transform(image=image)["image"].permute(1, 2, 0).cpu().numpy()
        aug_img = np.clip(aug_img, 0, 1)

        # Plot original vs augmented
        plt.figure(figsize=(12, 6))
        plt.subplot(1, 2, 1)
        plt.imshow(orig_img)
        plt.title("Original")
        plt.axis("off")

        plt.subplot(1, 2, 2)
        plt.imshow(aug_img)
        plt.title("Augmented")
        plt.axis("off")

        plt.show()

# Run it
show_augmented_images(image_paths)

### Paths Datasets and Dataloaders

In [ ]:
# Paths
train_img_dir = "/Users/snirtahasa/Desktop/Almonds - Project/Almons-Trees-3/train"
train_ann_path = os.path.join(train_img_dir, "_annotations.coco.json")
val_img_dir = "/Users/snirtahasa/Desktop/Almonds - Project/Almons-Trees-3/valid"
val_ann_path = os.path.join(val_img_dir, "_annotations.coco.json")

# Datasets and Dataloaders
train_dataset = COCOSegmentationDataset(train_img_dir, train_ann_path, train_transform)
val_dataset = COCOSegmentationDataset(val_img_dir, val_ann_path, val_transform)

train_loader = DataLoader(train_dataset, batch_size=1, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=1, shuffle=False, num_workers=0)

### Model

In [ ]:
# Model
model = smp.Unet(
    encoder_name="resnet34",
    encoder_weights="imagenet",
    in_channels=3,
    classes=1,
    activation="sigmoid"
).to(device)


In [ ]:
import torch.nn.functional as F

# Dice + BCE Loss function
class CombinedLoss(torch.nn.Module):
    def __init__(self, dice_weight=1.0, bce_weight=1.0):
        super().__init__()
        self.dice = smp.losses.DiceLoss(mode='binary')
        self.bce = torch.nn.BCELoss()
        self.dice_weight = dice_weight
        self.bce_weight = bce_weight

    def forward(self, inputs, targets):
        dice_loss = self.dice(inputs, targets)
        bce_loss = self.bce(inputs, targets)
        return self.dice_weight * dice_loss + self.bce_weight * bce_loss

# Loss
loss_fn = CombinedLoss(dice_weight=1.0, bce_weight=1.0)


# Optimizer
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

In [ ]:
def dice_coef(preds, targets, threshold=0.5, eps=1e-6):
    preds = (preds > threshold).float()
    intersection = (preds * targets).sum(dim=(1, 2, 3))
    union = preds.sum(dim=(1, 2, 3)) + targets.sum(dim=(1, 2, 3))
    dice = (2. * intersection + eps) / (union + eps)
    return dice.mean()

def iou_score(preds, targets, threshold=0.5, eps=1e-6):
    preds = (preds > threshold).float()
    intersection = (preds * targets).sum(dim=(1, 2, 3))
    union = preds.sum(dim=(1, 2, 3)) + targets.sum(dim=(1, 2, 3)) - intersection
    iou = (intersection + eps) / (union + eps)
    return iou.mean()

In [ ]:
import matplotlib.pyplot as plt

def plot_learning_curves(history, metrics=None):
    if metrics is None:
        metrics = ['train_loss', 'val_loss', 'train_dice', 'val_dice', 'train_iou', 'val_iou']
    
    plt.figure(figsize=(15, 10))
    for metric in metrics:
        plt.plot(history[metric], label=metric)

    plt.xlabel("Epoch")
    plt.ylabel("Score")
    plt.title("Training and Validation Curves")
    plt.legend()
    plt.grid(True)
    plt.show()

### Training and Evaluation

In [ ]:
# Initialize history tracking
history = {
    'train_loss': [],
    'val_loss': [],
    'train_dice': [],
    'val_dice': [],
    'train_iou': [],
    'val_iou': []
}

best_val_dice = 0

for epoch in range(1, 51):
    model.train()
    train_loss = 0
    train_dice = 0
    train_iou = 0
    for images, masks in tqdm(train_loader, desc=f"Epoch {epoch} [Train]"):
        images, masks = images.to(device), masks.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = loss_fn(outputs, masks)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
        train_dice += dice_coef(outputs, masks).item()
        train_iou += iou_score(outputs, masks).item()

    model.eval()
    val_loss = 0
    val_dice = 0
    val_iou = 0
    with torch.no_grad():
        for images, masks in tqdm(val_loader, desc=f"Epoch {epoch} [Val]"):
            images, masks = images.to(device), masks.to(device)
            outputs = model(images)
            val_loss += loss_fn(outputs, masks).item()
            val_dice += dice_coef(outputs, masks).item()
            val_iou += iou_score(outputs, masks).item()

    # Averages
    avg_train_loss = train_loss / len(train_loader)
    avg_val_loss = val_loss / len(val_loader)
    avg_train_dice = train_dice / len(train_loader)
    avg_val_dice = val_dice / len(val_loader)
    avg_train_iou = train_iou / len(train_loader)
    avg_val_iou = val_iou / len(val_loader)

    # Logging
    print(f"Epoch {epoch} | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | "
          f"Train Dice: {avg_train_dice:.4f} | Val Dice: {avg_val_dice:.4f} | "
          f"Train IoU: {avg_train_iou:.4f} | Val IoU: {avg_val_iou:.4f}")

    # Save best model
    if avg_val_dice > best_val_dice:
        best_val_dice = avg_val_dice
        torch.save(model.state_dict(), "best_model.pth")
        print("✅ Saved new best model")

    # Update history
    history['train_loss'].append(avg_train_loss)
    history['val_loss'].append(avg_val_loss)
    history['train_dice'].append(avg_train_dice)
    history['val_dice'].append(avg_val_dice)
    history['train_iou'].append(avg_train_iou)
    history['val_iou'].append(avg_val_iou)

In [ ]:
plot_learning_curves(history)

In [ ]:
# 1. Define test transform
test_transform = A.Compose([
    A.Resize(1024, 1024), 
    A.Normalize(),
    ToTensorV2(),
])

# 2. Load test dataset
test_dataset = COCOSegmentationDataset(
    img_dir="/Users/snirtahasa/Desktop/Almonds - Project/Almons-Trees-3/test/",  
    ann_path="/Users/snirtahasa/Desktop/Almonds - Project/Almons-Trees-3/test/_annotations.coco.json", 
    transform=test_transform
)

# 3. Load best model
model.load_state_dict(torch.load("best_model.pth", map_location=device))
model.eval()

# 4. Prediction function (same as before)
def tensor_to_image(tensor):
    tensor = tensor.squeeze().cpu().numpy()
    if tensor.ndim == 3:
        tensor = np.transpose(tensor, (1, 2, 0))
    tensor = (tensor * 255).astype(np.uint8)
    return tensor

def visualize_predictions(model, dataset, device, max_samples=None):
    n_samples = len(dataset) if max_samples is None else min(len(dataset), max_samples)
    for i in tqdm(range(n_samples), desc="Predicting"):
        image, true_mask = dataset[i]
        image_tensor = image.unsqueeze(0).to(device)
        with torch.no_grad():
            pred_mask = model(image_tensor)
        pred_mask = (pred_mask.squeeze().cpu().numpy() > 0.5).astype(np.uint8)
        image_np = tensor_to_image(image)
        true_mask_np = true_mask.squeeze().cpu().numpy()

        # Plot
        fig, axs = plt.subplots(1, 3, figsize=(12, 4))
        axs[0].imshow(image_np)
        axs[0].set_title("Original Image")
        axs[1].imshow(true_mask_np, cmap='gray')
        axs[1].set_title("Ground Truth")
        axs[2].imshow(pred_mask, cmap='gray')
        axs[2].set_title("Predicted Mask")
        for ax in axs:
            ax.axis('off')
        plt.tight_layout()
        plt.show()

# 5. Run predictions
visualize_predictions(model, test_dataset, device, max_samples=10) 
